# Error Table Utilities Layer Tests

Validation notebook for `errors.py`, the standalone MAD-X error-table utility layer.


## 1. Imports and setup

The notebook is intended to run from `Dev/04_Error_Table_Utilities/`. Generated files stay inside `./error_table_tests/`.


In [1]:
from pathlib import Path
import shutil
import sys

import numpy as np
import pandas as pd

PROJECT_DIR = Path.cwd()
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

from errors import (
    MAD_X_ERROR_COLUMNS,
    MAD_X_ERROR_NUMERIC_COLUMNS,
    normalise_error_table_columns,
    validate_error_table,
    read_error_table,
    write_error_table,
    combine_error_tables,
    filter_error_table,
    flip_error_columns,
    map_error_names_to_lattice,
    summarise_error_table,
    apply_error_table,
)
from cycle_time import RCSRamp
from machine_state import MachineState
from machine_state_writer import write_machine_state_file
from madx_model import MadxModel

LATTICE_FOLDER = "../Lattice_Files/00_Simplified_Lattice/"
SEQUENCE_NAME = "synchrotron"
OUTPUT_DIR = Path("./error_table_tests")

if OUTPUT_DIR.exists():
    shutil.rmtree(OUTPUT_DIR)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Project directory:", PROJECT_DIR)
print("Output directory:", OUTPUT_DIR.resolve())


Project directory: /home/hr/Repositories/optics_gui/Dev/04_Error_Table_Utilities
Output directory: /home/hr/Repositories/optics_gui/Dev/04_Error_Table_Utilities/error_table_tests


## 2. Synthetic DataFrame normalisation tests

Mixed-case MAD-X columns should normalise to lower-case pandas columns, with missing alignment columns filled as zero.


In [2]:
synthetic_mixed = pd.DataFrame({
    "NAME": ["sp0_qf", "sp1_qd"],
    "DX": [1.0e-3, 0.0],
    "DY": [0.0, -2.0e-3],
    "DTHETA": [0.0, 1.0e-4],
})

normalised = normalise_error_table_columns(synthetic_mixed)

assert list(normalised.columns) == list(MAD_X_ERROR_COLUMNS)
assert normalised["name"].tolist() == ["sp0_qf", "sp1_qd"]
for column in MAD_X_ERROR_NUMERIC_COLUMNS:
    assert np.issubdtype(normalised[column].dtype, np.floating), column

assert np.allclose(normalised["dx"], [1.0e-3, 0.0])
assert np.allclose(normalised["dy"], [0.0, -2.0e-3])
assert np.allclose(normalised["dtheta"], [0.0, 1.0e-4])
assert np.allclose(normalised["ds"], 0.0)
assert np.allclose(normalised["dphi"], 0.0)
assert np.allclose(normalised["dpsi"], 0.0)

try:
    normalise_error_table_columns(pd.DataFrame({"DX": [1.0], "dx": [2.0]}))
    raise AssertionError("Expected duplicate-column normalisation failure")
except ValueError as exc:
    print("Caught expected duplicate-column error:", exc)

normalised


Caught expected duplicate-column error: Duplicate columns produced by normalisation: ['dx']


,name,dx,dy,ds,dphi,dtheta,dpsi
0,sp0_qf,0.001,0.000,0.0,0.0,0.0000,0.0
1,sp1_qd,0.000,-0.002,0.0,0.0,0.0001,0.0


## 3. Validation tests

Valid tables should pass; malformed tables should fail clearly before they reach MAD-X.


In [3]:
valid_table = validate_error_table(normalised)
assert list(valid_table.columns) == list(MAD_X_ERROR_COLUMNS)

try:
    validate_error_table("not a dataframe")
    raise AssertionError("Expected TypeError for non-DataFrame input")
except TypeError as exc:
    print("Caught expected TypeError:", exc)

try:
    validate_error_table(pd.DataFrame({"DX": [0.0]}))
    raise AssertionError("Expected ValueError for missing name")
except ValueError as exc:
    print("Caught expected missing-name error:", exc)

try:
    validate_error_table(pd.DataFrame(columns=MAD_X_ERROR_COLUMNS))
    raise AssertionError("Expected ValueError for empty table")
except ValueError as exc:
    print("Caught expected empty-table error:", exc)

try:
    bad_names = normalised.copy()
    bad_names.loc[0, "name"] = ""
    validate_error_table(bad_names)
    raise AssertionError("Expected ValueError for empty name")
except ValueError as exc:
    print("Caught expected empty-name error:", exc)

try:
    bad_numeric = normalised.copy()
    bad_numeric.loc[0, "dx"] = np.inf
    validate_error_table(bad_numeric)
    raise AssertionError("Expected ValueError for non-finite numeric value")
except ValueError as exc:
    print("Caught expected non-finite error:", exc)

try:
    duplicate_names = pd.concat([normalised, normalised.iloc[[0]]], ignore_index=True)
    validate_error_table(duplicate_names, require_unique_names=True)
    raise AssertionError("Expected ValueError for duplicate names")
except ValueError as exc:
    print("Caught expected duplicate-name error:", exc)

try:
    extra = normalised.copy()
    extra["comment"] = "extra"
    validate_error_table(extra, allow_extra_columns=False)
    raise AssertionError("Expected ValueError for unexpected columns")
except ValueError as exc:
    print("Caught expected extra-column error:", exc)

print("Validation tests passed")


Caught expected TypeError: df must be a pandas DataFrame.
Caught expected missing-name error: Missing required error-table columns: ['name', 'dy', 'ds', 'dphi', 'dtheta', 'dpsi']
Caught expected empty-table error: Error table is empty.
Caught expected empty-name error: Error table contains null or empty names.
Caught expected non-finite error: Column 'dx' contains non-finite values.
Caught expected duplicate-name error: Error table contains duplicate names: ['sp0_qf']
Caught expected extra-column error: Unexpected error-table columns: ['comment']
Validation tests passed


## 4. Write/read round-trip tests

The writer should produce a MAD-X-readable table that the local reader can round-trip.


In [4]:
round_trip_path = OUTPUT_DIR / "synthetic_error_table.tfs"
written_path = write_error_table(normalised, round_trip_path)
read_back = read_error_table(written_path)

assert Path(written_path).is_file()
assert set(MAD_X_ERROR_COLUMNS).issubset(read_back.columns)
assert "k0l" in read_back.columns
assert read_back["name"].tolist() == normalised["name"].tolist()
for column in MAD_X_ERROR_NUMERIC_COLUMNS:
    assert np.allclose(read_back[column], normalised[column]), column

summary = summarise_error_table(read_back)
assert summary["n_rows"] == 2
assert summary["n_unique_names"] == 2
assert summary["n_nonzero_rows"] == 2
assert summary["max_abs_dx"] == 1.0e-3
assert summary["max_abs_dy"] == 2.0e-3

summary


{'n_rows': 2,
 'n_unique_names': 2,
 'n_nonzero_rows': 2,
 'columns': ['name',
  'dx',
  'dy',
  'ds',
  'dphi',
  'dtheta',
  'dpsi',
  'k0l',
  'k0sl',
  'k1l',
  'k1sl',
  'k2l',
  'k2sl',
  'k3l',
  'k3sl',
  'k4l',
  'k4sl',
  'k5l',
  'k5sl',
  'k6l',
  'k6sl',
  'k7l',
  'k7sl',
  'k8l',
  'k8sl',
  'k9l',
  'k9sl',
  'k10l',
  'k10sl',
  'k11l',
  'k11sl',
  'k12l',
  'k12sl',
  'k13l',
  'k13sl',
  'k14l',
  'k14sl',
  'k15l',
  'k15sl',
  'k16l',
  'k16sl',
  'k17l',
  'k17sl',
  'k18l',
  'k18sl',
  'k19l',
  'k19sl',
  'k20l',
  'k20sl',
  'mrex',
  'mrey',
  'mredx',
  'mredy',
  'arex',
  'arey',
  'mscalx',
  'mscaly',
  'rfm_freq',
  'rfm_harmon',
  'rfm_lag',
  'p0l',
  'p0sl',
  'p1l',
  'p1sl',
  'p2l',
  'p2sl',
  'p3l',
  'p3sl',
  'p4l',
  'p4sl',
  'p5l',
  'p5sl',
  'p6l',
  'p6sl',
  'p7l',
  'p7sl',
  'p8l',
  'p8sl',
  'p9l',
  'p9sl',
  'p10l',
  'p10sl',
  'p11l',
  'p11sl',
  'p12l',
  'p12sl',
  'p13l',
  'p13sl',
  'p14l',
  'p14sl',
  'p15l',
  'p15sl',

## 5. Combine tests

Combination is currently defined as summing numeric error columns by element name, preserving first-seen order.


In [5]:
df1 = pd.DataFrame({
    "NAME": ["sp0_qf", "sp1_qd"],
    "DX": [1.0e-3, 2.0e-3],
})
df2 = pd.DataFrame({
    "name": ["sp1_qd", "sp2_qf"],
    "dy": [3.0e-3, 4.0e-3],
    "dx": [5.0e-3, 6.0e-3],
})

combined = combine_error_tables([df1, df2], mode="sum")
assert combined["name"].tolist() == ["sp0_qf", "sp1_qd", "sp2_qf"]
assert np.isclose(combined.loc[combined["name"] == "sp0_qf", "dx"].iloc[0], 1.0e-3)
assert np.isclose(combined.loc[combined["name"] == "sp1_qd", "dx"].iloc[0], 7.0e-3)
assert np.isclose(combined.loc[combined["name"] == "sp1_qd", "dy"].iloc[0], 3.0e-3)
assert np.isclose(combined.loc[combined["name"] == "sp2_qf", "dy"].iloc[0], 4.0e-3)

combined_from_path = combine_error_tables([df1, written_path], mode="sum")
assert "sp0_qf" in combined_from_path["name"].tolist()

try:
    combine_error_tables([], mode="sum")
    raise AssertionError("Expected ValueError for empty combine input")
except ValueError as exc:
    print("Caught expected empty-combine error:", exc)

try:
    combine_error_tables([df1, df2], mode="replace")
    raise AssertionError("Expected ValueError for unsupported combine mode")
except ValueError as exc:
    print("Caught expected unsupported-mode error:", exc)

combined


Caught expected empty-combine error: At least one error table is required.
Caught expected unsupported-mode error: Unsupported combine mode: 'replace'


,name,dx,dy,ds,dphi,dtheta,dpsi
0,sp0_qf,0.001,0.000,0.0,0.0,0.0,0.0
1,sp1_qd,0.007,0.003,0.0,0.0,0.0,0.0
2,sp2_qf,0.006,0.004,0.0,0.0,0.0,0.0


## 6. Filter tests

Filters should work by explicit names, regex, substring, case-insensitive matching, inversion, and optional element type.


In [6]:
filter_source = normalised.copy()
filter_source["element_type"] = ["quadrupole", "quadrupole"]
filter_source.loc[len(filter_source)] = {
    "name": "SP2_DIP",
    "dx": 0.0,
    "dy": 0.0,
    "ds": 0.0,
    "dphi": 0.0,
    "dtheta": 0.0,
    "dpsi": 0.0,
    "element_type": "sbend",
}

by_name = filter_error_table(filter_source, names=["sp0_qf"])
assert by_name["name"].tolist() == ["sp0_qf"]

by_regex = filter_error_table(filter_source, patterns=[r"sp[01]_q[fd]"])
assert by_regex["name"].tolist() == ["sp0_qf", "sp1_qd"]

by_substring = filter_error_table(filter_source, patterns=["dip"], regex=False)
assert by_substring["name"].tolist() == ["SP2_DIP"]

case_insensitive = filter_error_table(filter_source, names=["sp2_dip"], case=False)
assert case_insensitive["name"].tolist() == ["SP2_DIP"]

inverted = filter_error_table(filter_source, patterns=["q"], regex=False, invert=True)
assert inverted["name"].tolist() == ["SP2_DIP"]

by_type = filter_error_table(filter_source, element_types=["SBEND"], case=False)
assert by_type["name"].tolist() == ["SP2_DIP"]

unchanged = filter_error_table(filter_source)
assert unchanged["name"].tolist() == filter_source["name"].tolist()

print("Filter tests passed")


Filter tests passed


## 7. Flip tests

Selected numeric error columns should change sign without affecting other columns.


In [7]:
flipped_dx = flip_error_columns(normalised, ["dx"])
assert np.allclose(flipped_dx["dx"], -normalised["dx"])
assert np.allclose(flipped_dx["dy"], normalised["dy"])

flipped_multiple = flip_error_columns(normalised, ["DX", "DTHETA"])
assert np.allclose(flipped_multiple["dx"], -normalised["dx"])
assert np.allclose(flipped_multiple["dtheta"], -normalised["dtheta"])

try:
    flip_error_columns(normalised, ["name"])
    raise AssertionError("Expected ValueError for flipping name")
except ValueError as exc:
    print("Caught expected name-flip error:", exc)

try:
    extra = normalised.copy()
    extra["comment"] = "not an error column"
    flip_error_columns(extra, ["comment"])
    raise AssertionError("Expected ValueError for non-error column")
except ValueError as exc:
    print("Caught expected non-error-column flip error:", exc)

print("Flip tests passed")


Caught expected name-flip error: Can only flip numeric MAD-X error columns: ['name']
Caught expected non-error-column flip error: Can only flip numeric MAD-X error columns: ['comment']
Flip tests passed


## 8. Lattice-name mapping tests

Mapping should retain unmatched error-table names and mark them explicitly.


In [8]:
synthetic_twiss_df = pd.DataFrame({"name": ["sp0_qf", "sp1_qd", "sp2_qf"]})
map_source = pd.DataFrame({
    "NAME": ["sp0_qf", "sp9_missing", "SP1_QD"],
    "DX": [1.0e-3, 2.0e-3, 3.0e-3],
})

mapped = map_error_names_to_lattice(map_source, synthetic_twiss_df)
assert mapped.loc[mapped["name"] == "sp0_qf", "in_lattice"].iloc[0] is True or bool(mapped.loc[mapped["name"] == "sp0_qf", "in_lattice"].iloc[0]) is True
assert bool(mapped.loc[mapped["name"] == "sp9_missing", "in_lattice"].iloc[0]) is False
assert pd.isna(mapped.loc[mapped["name"] == "sp9_missing", "matched_name"].iloc[0])
assert bool(mapped.loc[mapped["name"] == "SP1_QD", "in_lattice"].iloc[0]) is True

case_sensitive = map_error_names_to_lattice(map_source, synthetic_twiss_df, case=True)
assert bool(case_sensitive.loc[case_sensitive["name"] == "SP1_QD", "in_lattice"].iloc[0]) is False

try:
    map_error_names_to_lattice(map_source, pd.DataFrame({"element": ["sp0_qf"]}))
    raise AssertionError("Expected ValueError for missing TWISS name column")
except ValueError as exc:
    print("Caught expected TWISS-column error:", exc)

mapped


Caught expected TWISS-column error: TWISS DataFrame is missing lattice name column 'name'.


,name,matched_name,in_lattice,dx,dy,ds,dphi,dtheta,dpsi
0,sp0_qf,sp0_qf,True,0.001,0.0,0.0,0.0,0.0,0.0
1,sp9_missing,None,False,0.002,0.0,0.0,0.0,0.0,0.0
2,SP1_QD,sp1_qd,True,0.003,0.0,0.0,0.0,0.0,0.0


## 9. Integration with `MadxModel.apply_error_table(...)`

This section uses the copied current sources and real MAD-X through `MadxModel`. It does not call `USE` after applying the error table.


In [9]:
ramp = RCSRamp(top_energy_MeV=800.0)
beam_state = ramp.state_at(3.0)
machine_state = MachineState.from_defaults(
    beam_state=beam_state,
    main_magnet_mode="srm_bare",
    requested_qx=4.295,
    requested_qy=3.825,
    tune_method="di_wright",
)

state_file = write_machine_state_file(
    machine_state,
    output_dir=OUTPUT_DIR / "machine_states",
    filename="machine_state_default.strength",
)

model_default = MadxModel(
    lattice_folder=LATTICE_FOLDER,
    sequence_name=SEQUENCE_NAME,
    machine_state_file=state_file,
    aperture_file="ISIS.aperture",
    output_dir=OUTPUT_DIR / "madx_default",
)
model_default.load_lattice(use_sequence=True)
twiss_df_default = model_default.run_twiss()
summary_df_default = model_default.get_summary_df()

assert not twiss_df_default.empty
assert not summary_df_default.empty
assert {"name", "x", "y"}.issubset(twiss_df_default.columns)

real_mapping = map_error_names_to_lattice(normalised, twiss_df_default)
assert "in_lattice" in real_mapping.columns

# Target a real QD quadrupole, not a fringe element.
name_series = twiss_df_default["name"].astype(str)
base_names = name_series.str.split(":").str[0].str.lower()
candidates = twiss_df_default[
    name_series.notna()
    & (name_series != "")
    & base_names.str.endswith("_qd")
]
assert not candidates.empty, "No QD elements found in TWISS table"

target_name = str(candidates.iloc[0]["name"]).split(":")[0]
assert target_name.lower().endswith("_qd")
integration_error_df = pd.DataFrame({
    "name": [target_name],
    "dx": [0.0],
    "dy": [5.0e-3],
    "ds": [0.0],
    "dphi": [0.0],
    "dtheta": [0.0],
    "dpsi": [0.0],
})

integration_error_path = write_error_table(
    integration_error_df,
    OUTPUT_DIR / "integration_error_table.tfs",
)

model_misaligned = MadxModel(
    lattice_folder=LATTICE_FOLDER,
    sequence_name=SEQUENCE_NAME,
    machine_state_file=state_file,
    aperture_file="ISIS.aperture",
    output_dir=OUTPUT_DIR / "madx_misaligned",
)
model_misaligned.load_lattice(use_sequence=True)
model_misaligned.apply_error_table(integration_error_path)
twiss_df_misaligned = model_misaligned.run_twiss()

assert model_misaligned.metadata["errors_applied"] is True
assert model_misaligned.metadata["error_table_path"] == integration_error_path
assert integration_error_path in model_misaligned.loaded_files
assert not twiss_df_misaligned.empty

dx_delta = float(np.max(np.abs(twiss_df_misaligned["x"].to_numpy() - twiss_df_default["x"].to_numpy())))
dy_delta = float(np.max(np.abs(twiss_df_misaligned["y"].to_numpy() - twiss_df_default["y"].to_numpy())))
max_orbit_delta = max(dx_delta, dy_delta)
assert np.isfinite(dx_delta)
assert np.isfinite(dy_delta)
assert max_orbit_delta >= 1.0e-3, (
    "Expected a mm-level orbit change from a 5 mm QD vertical offset; "
    f"got dx={dx_delta:.6g} m, dy={dy_delta:.6g} m"
)

print("Integration target:", target_name)
print("Horizontal orbit delta:", dx_delta)
print("Vertical orbit delta:", dy_delta)
print("Max orbit delta:", max_orbit_delta)


Integration target: sp0_qd
Horizontal orbit delta: 4.376431360305483e-05
Vertical orbit delta: 0.02605315023574523
Max orbit delta: 0.02605315023574523


## 10. Single-element orbit response matrix

Apply one MAD-X alignment error at a time to representative QD, QF, QDS, and DIP elements. Observable cases should move the closed orbit by millimetres; quadrupole `ds` and `dpsi` are valid table entries but produce no closed-orbit change in this lattice, so those cases assert stable zero response after successful loading.


In [10]:
MIN_OBSERVABLE_M = 1.0e-3
MAX_REASONABLE_M = 8.0e-2
ZERO_RESPONSE_TOL_M = 1.0e-9

TARGET_ELEMENTS = {
    "QD": "sp0_qd:1",
    "QF": "sp0_qf:1",
    "QDS": "sp0_qds:1",
    "DIP": "sp1_dip:1",
}

for family, element_name in TARGET_ELEMENTS.items():
    assert element_name in set(twiss_df_default["name"].astype(str)), element_name

OBSERVABLE_ERROR_CASES = [
    ("QD", "dx", 1.0e-3),
    ("QD", "dy", 3.0e-4),
    ("QD", "dphi", 1.0e-3),
    ("QD", "dtheta", 3.0e-3),
    ("QF", "dx", 3.0e-4),
    ("QF", "dy", 1.0e-3),
    ("QF", "dphi", 3.0e-3),
    ("QF", "dtheta", 3.0e-3),
    ("QDS", "dx", 3.0e-3),
    ("QDS", "dy", 1.0e-3),
    ("QDS", "dphi", 1.0e-2),
    ("QDS", "dtheta", 3.0e-2),
    ("DIP", "dx", 1.0e-3),
    ("DIP", "dy", 1.0e-3),
    ("DIP", "ds", 3.0e-3),
    ("DIP", "dphi", 3.0e-4),
    ("DIP", "dtheta", 3.0e-4),
    ("DIP", "dpsi", 3.0e-4),
]

EXPECTED_ZERO_RESPONSE_CASES = [
    (family, error_column, 1.0e-1)
    for family in ("QD", "QF", "QDS")
    for error_column in ("ds", "dpsi")
]


def run_single_error_response(family, error_column, error_value):
    element_name = TARGET_ELEMENTS[family]
    error_row = {
        "name": element_name,
        "dx": 0.0,
        "dy": 0.0,
        "ds": 0.0,
        "dphi": 0.0,
        "dtheta": 0.0,
        "dpsi": 0.0,
    }
    error_row[error_column] = error_value

    case_name = f"{family.lower()}_{error_column}"
    error_path = write_error_table(
        pd.DataFrame([error_row]),
        OUTPUT_DIR / f"single_{case_name}.tfs",
        table_name=f"SINGLE_{family}_{error_column}".upper(),
    )

    model = MadxModel(
        lattice_folder=LATTICE_FOLDER,
        sequence_name=SEQUENCE_NAME,
        machine_state_file=state_file,
        aperture_file="ISIS.aperture",
        output_dir=OUTPUT_DIR / f"madx_single_{case_name}",
    )
    model.load_lattice(use_sequence=True)
    returned_path = model.apply_error_table(error_path)
    assert returned_path == error_path
    assert model.metadata["errors_applied"] is True
    assert model.metadata["error_table_path"] == error_path
    assert error_path in model.loaded_files

    twiss_df = model.run_twiss()
    assert not twiss_df.empty

    delta_x = twiss_df["x"].to_numpy() - twiss_df_default["x"].to_numpy()
    delta_y = twiss_df["y"].to_numpy() - twiss_df_default["y"].to_numpy()
    max_abs_x = float(np.nanmax(np.abs(delta_x)))
    max_abs_y = float(np.nanmax(np.abs(delta_y)))
    max_orbit = float(np.nanmax(np.sqrt(delta_x * delta_x + delta_y * delta_y)))

    return {
        "family": family,
        "element": element_name,
        "error_column": error_column,
        "error_value": error_value,
        "max_abs_x_m": max_abs_x,
        "max_abs_y_m": max_abs_y,
        "max_orbit_m": max_orbit,
        "error_path": error_path,
    }


response_rows = []
for family, error_column, error_value in OBSERVABLE_ERROR_CASES:
    result = run_single_error_response(family, error_column, error_value)
    assert MIN_OBSERVABLE_M <= result["max_orbit_m"] <= MAX_REASONABLE_M, result
    response_rows.append({**result, "expected": "observable"})

for family, error_column, error_value in EXPECTED_ZERO_RESPONSE_CASES:
    result = run_single_error_response(family, error_column, error_value)
    assert result["max_orbit_m"] <= ZERO_RESPONSE_TOL_M, result
    response_rows.append({**result, "expected": "zero_orbit_response"})

response_matrix_df = pd.DataFrame(response_rows)
response_matrix_df["max_orbit_mm"] = 1.0e3 * response_matrix_df["max_orbit_m"]
response_matrix_df["max_abs_x_mm"] = 1.0e3 * response_matrix_df["max_abs_x_m"]
response_matrix_df["max_abs_y_mm"] = 1.0e3 * response_matrix_df["max_abs_y_m"]

assert len(response_matrix_df) == len(OBSERVABLE_ERROR_CASES) + len(EXPECTED_ZERO_RESPONSE_CASES)
assert set(response_matrix_df["family"]) == set(TARGET_ELEMENTS)
assert set(response_matrix_df["error_column"]) == set(MAD_X_ERROR_NUMERIC_COLUMNS)

print(
    response_matrix_df[
        [
            "family",
            "element",
            "error_column",
            "error_value",
            "expected",
            "max_orbit_mm",
            "max_abs_x_mm",
            "max_abs_y_mm",
        ]
    ].to_string(index=False)
)


family   element error_column  error_value            expected  max_orbit_mm  max_abs_x_mm  max_abs_y_mm
    QD  sp0_qd:1           dx       0.0010          observable      2.292435      2.292435      0.000000
    QD  sp0_qd:1           dy       0.0003          observable      1.563201      0.000158      1.563201
    QD  sp0_qd:1         dphi       0.0010          observable      1.342088      0.000116      1.342088
    QD  sp0_qd:1       dtheta       0.0030          observable      1.817377      1.817377      0.000000
    QF  sp0_qf:1           dx       0.0003          observable      1.023272      1.023272      0.000000
    QF  sp0_qf:1           dy       0.0010          observable      3.202903      0.000667      3.202903
    QF  sp0_qf:1         dphi       0.0030          observable      2.371166      0.000365      2.371166
    QF  sp0_qf:1       dtheta       0.0030          observable      2.596481      2.596481      0.000000
   QDS sp0_qds:1           dx       0.0030          obs

## 11. Thin wrapper test

`errors.apply_error_table(...)` should delegate to `MadxModel.apply_error_table(...)` and preserve metadata behaviour.


In [11]:
model_wrapper = MadxModel(
    lattice_folder=LATTICE_FOLDER,
    sequence_name=SEQUENCE_NAME,
    machine_state_file=state_file,
    aperture_file="ISIS.aperture",
    output_dir=OUTPUT_DIR / "madx_wrapper",
)
model_wrapper.load_lattice(use_sequence=True)
returned_path = apply_error_table(model_wrapper, integration_error_path)
assert returned_path == integration_error_path
assert model_wrapper.metadata["errors_applied"] is True
assert model_wrapper.metadata["error_table_path"] == integration_error_path
assert integration_error_path in model_wrapper.loaded_files

twiss_df_wrapper = model_wrapper.run_twiss()
assert not twiss_df_wrapper.empty

print("Thin wrapper test passed")


Thin wrapper test passed


## 12. Negative integration tests

Missing files and invalid tables should fail before any silent MAD-X behaviour.


In [12]:
try:
    read_error_table(OUTPUT_DIR / "missing_file.tfs")
    raise AssertionError("Expected FileNotFoundError for missing error table")
except FileNotFoundError as exc:
    print("Caught expected read_error_table error:", exc)

try:
    model_wrapper.apply_error_table(OUTPUT_DIR / "missing_file.tfs")
    raise AssertionError("Expected FileNotFoundError for missing MadxModel error table")
except FileNotFoundError as exc:
    print("Caught expected MadxModel error-table error:", exc)

invalid_table = pd.DataFrame({"name": ["sp0_qf"], "dx": [np.nan]})
try:
    validate_error_table(invalid_table, required_columns=("name", "dx"))
    raise AssertionError("Expected invalid table validation failure")
except ValueError as exc:
    print("Caught expected invalid-table error:", exc)

print("Negative integration tests passed")


Caught expected read_error_table error: Missing error table: error_table_tests/missing_file.tfs
Caught expected MadxModel error-table error: Missing MAD-X error table: error_table_tests/missing_file.tfs
Caught expected invalid-table error: Column 'dx' contains non-finite values.
Negative integration tests passed


## 13. Placeholders / missing details

No public functions are placeholders in this layer. ISIS-specific physics beyond DataFrame/file handling remains outside `errors.py` until the relevant convention is agreed.


## 14. Final status


In [13]:
print("errors.py layer validation complete")


errors.py layer validation complete
